# **Urban Cycling Route Planner**

In [1]:
from ipyleaflet import Map, Marker
from ipywidgets import Button, Output, VBox

from data_handler import fill_data
from algorithms import dijkstra, a_star
from algorithms_networkx import convert_to_networkx, dijkstra_networkx, astar_networkx, betweenness_centrality
import data_structure as ds
from visualize import draw_route, draw_points
from helpers import closest_node_calculation

In [2]:
area = "Budapest"
network = fill_data(area)

if network == "error":
    print("Error fetching data")

graph = convert_to_networkx(network)

In [ ]:
m = Map(center=(47.4979, 19.0402), zoom=13)
button = Button(description="Find route", button_style="success", disabled=True)
out = Output()

coords = []
def handle_click(**kwargs):
    if kwargs.get('type') == 'click':
        active_markers = [layer for layer in m.layers if isinstance(layer, Marker)]
        latlon = kwargs.get('coordinates')

        if len(active_markers) < 2:
            marker = Marker(location=latlon)
            m.add_layer(marker)
        else:
            m.remove_layer(active_markers[0])
            marker = Marker(location=latlon)
            m.add_layer(marker)

        active_markers = len([layer for layer in m.layers if isinstance(layer, Marker)])
        button.disabled = (active_markers != 2)
def handle_button_click(b):
    with out:
        out.clear_output()
        markers = [layer for layer in m.layers if isinstance(layer, Marker)]
        start_coords = markers[0].location
        end_coords = markers[1].location

        start_node = closest_node_calculation(network, start_coords[0], start_coords[1])
        end_node = closest_node_calculation(network, end_coords[0], end_coords[1])

        
        path, dist = dijkstra_networkx(graph, start_node, end_node)

        draw_route(network, path)

button.on_click(handle_button_click)        
m.on_interaction(handle_click)

VBox([m, button, out])